# Case study: a data quality report for the Berlin Airbnb listings

*Session 4, block 1 · Author: course team · Licence: CC-BY-4.0*

**Why.** A city housing analyst wants to report what a night costs in each district and how many listings carry a
registration number. Before a single median is computed, someone has to say whether the listings can be trusted:
prices stored as text, placeholder values, impossible minimum stays, names typed into the registration field,
listings that appear more than once. This notebook is that someone.

**Task.** Write a data quality report: which checks did you run, what did they find, how severe is each finding,
and what should happen next?

Every check is code, so that the report can be regenerated when Inside Airbnb publishes the next snapshot.
Sections 2–5 check the prepared tables (`listings`, `calendar`, `reviews_monthly`), section 6 looks at the **raw
file**, where some problems are still visible that `prepare_airbnb.py` already handled, section 8 writes the rules
as a **pandera** schema, and [`quality/test_listings_quality.py`](quality/test_listings_quality.py) contains them as
**pytest** tests.

Data: Inside Airbnb, Berlin snapshot of 26 June 2026 (CC BY 4.0), prepared with
`uv run python case-study/prepare_airbnb.py`. Cells marked **TODO** are for you.

In [ ]:
import sys
from pathlib import Path

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DATA = ROOT / "case-study" / "data" / "airbnb"
RAW = ROOT / "case-study" / "data" / "raw" / "airbnb"
assert (DATA / "listings.parquet").exists(), "run case-study/prepare_airbnb.py first"

import numpy as np
import pandas as pd

listings = pd.read_parquet(DATA / "listings.parquet")
calendar = pd.read_parquet(DATA / "calendar.parquet")
reviews_monthly = pd.read_parquet(DATA / "reviews_monthly.parquet")
print(listings.shape, calendar.shape, reviews_monthly.shape, "snapshot", (DATA / "snapshot.txt").read_text())

## 1. A small framework: every check returns a row

Each check records the **dimension** of data quality it belongs to, the table and column, the number of
failing rows and a **severity**:

- *critical*: breaks a key or a hard rule; the data cannot be used as they are;
- *major*: affects many rows or a central variable; must be handled before analysis;
- *minor*: worth documenting; handle when it matters for a question.

A check that passes is kept in the report with severity "ok": the next reader then knows it was run.

In [ ]:
results = []


def check(dimension, table, column, description, failed, n_total, severity):
    """Record one check. `failed` is a boolean Series (True = row fails) or a count."""
    n_failed = int(failed.sum()) if hasattr(failed, "sum") else int(failed)
    results.append({"dimension": dimension, "table": table, "column": column, "check": description,
                    "n_failed": n_failed, "share": n_failed / n_total,
                    "severity": severity if n_failed else "ok"})

## 2. Completeness: missing values, empty columns and placeholders

A value can be missing as `NULL`/`NaN` or *hidden* as a placeholder, such as the maximum stay 2,147,483,647 (the
largest 32-bit integer). Some values are missing **by design**: a listing without reviews has no rating. This is
called **structural missingness**; it is not a defect, but it must be documented and must not be imputed.

In [ ]:
n = len(listings)
REQUIRED = ["id", "host_id", "name", "district", "neighbourhood", "latitude", "longitude", "room_type",
            "accommodates", "availability_365", "number_of_reviews", "license_status"]
for col in REQUIRED:
    check("completeness", "listings", col, "value is NULL", listings[col].isna(), n, "critical")
for col, severity in [("price", "major"), ("bedrooms", "major"), ("beds", "minor"), ("bathrooms", "minor")]:
    check("completeness", "listings", col, "value is NULL", listings[col].isna(), n, severity)
check("completeness", "listings", "maximum_nights", "placeholder 2,147,483,647 (hidden missing)",
      listings["maximum_nights"].eq(2**31 - 1), n, "minor")
listings.isna().mean().sort_values(ascending=False).head(8).round(3)

In [ ]:
# structural missingness: who has a rating?
pd.crosstab(listings["number_of_reviews"].eq(0).rename("no reviews"),
            listings["review_scores_rating"].isna().rename("rating missing"), margins=True)

Every listing without reviews has no rating, and every listing with reviews has one: the missing ratings are
structural. The missing prices are a different matter, and block 2 of this session (workbook 08) asks whether they
are related to availability. A first look: 3,488 of the 4,335 listings without a price have no free night in the
next year, and almost every listing without a free night shows no price.

In [ ]:
pd.crosstab(listings["price"].isna().rename("price missing"),
            listings["availability_365"].eq(0).rename("no free night in 365 days"), margins=True)

In [ ]:
# columns of the Inside Airbnb file format that carry no value in this snapshot
raw = pd.read_csv(RAW / "listings.csv.gz", usecols=lambda c: c != "license")   # the raw registration field is not loaded
empty = raw.columns[raw.isna().all()].tolist()
check("completeness", "raw listings", ", ".join(empty[:3]) + ", ...", "column completely empty", len(empty),
      raw.shape[1], "minor")
print(len(empty), "empty columns:", empty)

Twelve columns exist in the file format but are empty in this snapshot, among them `host_since`, the response
and acceptance rates of hosts and `instant_bookable`. `prepare_airbnb.py` drops them. A question such as "do hosts
who have been active longer charge more?" cannot be answered with this snapshot: a completeness finding that no
imputation can fix.

## 3. Validity: types, domains and ranges

In [ ]:
expected_types = {"id": "int64", "price": "float64", "latitude": "float64", "longitude": "float64",
                  "accommodates": "int64", "first_review": "datetime64[ns]", "last_review": "datetime64[ns]"}
for col, dtype in expected_types.items():
    check("validity", "listings", col, f"dtype is {dtype}", int(str(listings[col].dtype) != dtype), 1, "critical")

ROOM_TYPES = {"Entire home/apt", "Private room", "Hotel room", "Shared room"}
LAT, LON = (52.33, 52.68), (13.08, 13.77)      # bounding box of neighbourhoods.geojson
check("validity", "listings", "room_type", "one of the four room types", ~listings["room_type"].isin(ROOM_TYPES), n, "critical")
check("validity", "listings", "latitude, longitude", "inside Berlin's bounding box",
      ~listings["latitude"].between(*LAT) | ~listings["longitude"].between(*LON), n, "critical")
check("validity", "listings", "accommodates", "at least one guest", listings["accommodates"].lt(1), n, "critical")
check("validity", "listings", "price", "price above 0", listings["price"].le(0), n, "critical")
check("validity", "listings", "price", "price below 10 EUR or above 1,000 EUR a night",
      ~listings["price"].between(10, 1000) & listings["price"].notna(), n, "minor")
check("validity", "listings", "minimum_nights", "minimum stay at most 365 nights", listings["minimum_nights"].gt(365), n, "minor")
check("validity", "listings", "maximum_nights", "maximum stay is not the 32-bit placeholder",
      listings["maximum_nights"].eq(2**31 - 1), n, "minor")
for col, days in [("availability_30", 30), ("availability_90", 90), ("availability_365", 365)]:
    check("validity", "listings", col, f"between 0 and {days} days", ~listings[col].between(0, days), n, "critical")
check("validity", "listings", "review_scores_rating", "rating between 0 and 5",
      ~listings["review_scores_rating"].between(0, 5) & listings["review_scores_rating"].notna(), n, "critical")

In [ ]:
# what do the stays look like?
print("most frequent maximum stays:", listings["maximum_nights"].value_counts().head(4).to_dict())
print("minimum stays above a year:", listings.loc[listings["minimum_nights"].gt(365), "minimum_nights"].value_counts().sort_index().to_dict())
print("listings with a minimum stay of 28 nights or more:", listings["minimum_nights"].ge(28).sum())

The most frequent maximum stay, 1,125 nights, is the value Airbnb fills in when a host sets no maximum: valid, but
it means "no limit chosen". The two values of 2,147,483,647 are software defaults, not stays. Minimum stays of
1,000 nights and more are allowed by the platform, but they are not holiday rentals. And a third of all listings
require 28 nights or more: Berlin restricts short-term rentals, and many hosts offer medium-term stays instead.
Their price field is not comparable with that of a short stay, which is why the course compares prices of short
stays only (`minimum_nights < 28`).

## 4. Uniqueness: keys and listings that repeat

In [ ]:
check("uniqueness", "listings", "id", "id unique", listings["id"].duplicated(), n, "critical")
SAME_OFFER = ["host_id", "name", "room_type", "accommodates"]
check("uniqueness", "listings", ", ".join(SAME_OFFER), "same host, title, room type and guests as another listing",
      listings.duplicated(SAME_OFFER), n, "minor")
check("uniqueness", "listings", "latitude, longitude", "exactly the same coordinates as another listing",
      listings.duplicated(["latitude", "longitude"]), n, "minor")

groups = listings[listings.duplicated(SAME_OFFER, keep=False)]
per_group = groups.groupby(SAME_OFFER).agg(n=("id", "size"), n_prices=("price", "nunique"),
                                           n_locations=("latitude", "nunique"))
print(len(groups), "listings in", len(per_group), "groups of", groups["host_id"].nunique(), "hosts")
print("groups at more than one location:", (per_group["n_locations"] > 1).sum())
per_group.reset_index(drop=True).sort_values("n", ascending=False).head(5)

No `id` repeats, so the key holds. But 473 listings of 61 hosts form groups with the same title, room type and
number of guests. Most groups are at different coordinates (Inside Airbnb also shifts each location by up to about
150 m to protect the host), and their prices often differ: these are operators of serviced apartments, hostels and
rental blocks who offer several similar units, not copies of one row. They are kept. Two consequences follow:
a count of "listings" is a count of offers, not of flats; and a price model validated with some units of one host in
training and others in the test set will look better than it is (Session 7).

## 5. Consistency: within the listings and across tables

In [ ]:
both = listings["minimum_nights"].notna() & listings["maximum_nights"].notna()
check("consistency", "listings", "minimum_nights", "minimum stay not above maximum stay",
      both & listings["minimum_nights"].gt(listings["maximum_nights"]), n, "critical")
check("consistency", "listings", "bedrooms", "not more bedrooms than guests",
      listings["bedrooms"].gt(listings["accommodates"]), n, "minor")
check("consistency", "listings", "first_review", "first review not after last review",
      listings["first_review"].gt(listings["last_review"]), n, "critical")
check("consistency", "listings", "review_scores_rating", "rating missing exactly when there is no review",
      listings["review_scores_rating"].isna().ne(listings["number_of_reviews"].eq(0)), n, "critical")
check("consistency", "listings", "availability_30", "free nights in 30 days not above those in 90 days",
      listings["availability_30"].gt(listings["availability_90"]), n, "critical")

In [ ]:
# across tables: do the review counts agree with the monthly review table?
monthly_total = reviews_monthly.groupby("listing_id")["n_reviews"].sum()
total = listings.set_index("id")["number_of_reviews"]
check("consistency", "listings / reviews_monthly", "number_of_reviews", "equals the sum of the monthly counts",
      total.ne(monthly_total.reindex(total.index, fill_value=0)), n, "critical")

# no review month before the first review, none after the last
months = reviews_monthly.join(listings.set_index("id")[["first_review", "last_review"]], on="listing_id", how="inner")
first_m = months["first_review"].dt.to_period("M").dt.to_timestamp()
last_m = months["last_review"].dt.to_period("M").dt.to_timestamp()
check("consistency", "reviews_monthly", "month", "no review month before the first review",
      months["month"].lt(first_m), len(months), "critical")
check("consistency", "reviews_monthly", "month", "no review month after the last review",
      months["month"].gt(last_m), len(months), "critical")

# foreign keys: every listing in calendar and reviews_monthly exists in listings
cal_ids = calendar["listing_id"].drop_duplicates()
rev_ids = reviews_monthly["listing_id"].drop_duplicates()
check("consistency", "calendar", "listing_id", "listing exists in listings (foreign key)",
      ~cal_ids.isin(listings["id"]), len(cal_ids), "major")
check("consistency", "reviews_monthly", "listing_id", "listing exists in listings (foreign key)",
      ~rev_ids.isin(listings["id"]), len(rev_ids), "major")
print("calendar rows of unknown listings:", (~calendar["listing_id"].isin(listings["id"])).sum())
print("review months after the snapshot month:", reviews_monthly["month"].gt("2026-06-01").sum())

The review counts and dates agree perfectly with the monthly table, which is no surprise: both come from the same
review file. The foreign keys do not hold: the calendar contains 79 listings and the review table 57 listings that
are not in the listings table. Inside Airbnb scrapes the three files separately (the listings between 26 June and
3 July 2026), so a listing can disappear between two scrapes. An inner join with the listings drops these rows
silently; a database with a foreign key would reject them (Session 3). The 22 review months after June 2026 are
reviews written while the scrape was still running: July 2026 is an incomplete month and must not be read as a drop
in demand (Session 12).

**TODO 1.** The calendar also has a column `minimum_nights` per date. In how many calendar rows does it differ from
the listing's `minimum_nights`? Is that a contradiction, or can a host set different minimum stays for different
dates?

In [ ]:
# TODO 1

## 6. The raw file: prices as text and names in the registration field

`prepare_airbnb.py` already converted the prices and removed personal data. The raw file shows what it had to deal
with. We read only the columns we need.

In [ ]:
raw_price = pd.read_csv(RAW / "listings.csv.gz", usecols=["price"])["price"]
print(raw_price.dtype, raw_price.dropna().head(3).tolist())
print("prices with a thousands separator:", raw_price.str.contains(",", na=False).sum())

as_number = pd.to_numeric(raw_price, errors="coerce")
check("validity", "raw listings", "price", "price is a number", as_number.isna() & raw_price.notna(), len(raw_price), "critical")

# the explicit rule of prepare_airbnb.py: remove the currency sign and the thousands separator, then convert
parsed = pd.to_numeric(raw_price.str.replace(r"[$,]", "", regex=True), errors="coerce")
print("parsed:", parsed.notna().sum(), "of", raw_price.notna().sum(), "| equal to the prepared table:",
      np.allclose(parsed.dropna().to_numpy(), listings["price"].dropna().to_numpy()))

Every price in the raw file is text such as `"$1,293.49"`, so `pd.to_numeric` fails on all 8,441 of them. The sign
is a dollar sign, although Berlin prices are in euros: Inside Airbnb writes `$` for every city. The rule "remove `$`
and `,`, then convert" parses every value, and the result equals the prepared table. Written as code, the rule
documents itself; a manual "Find and Replace" in a spreadsheet would not.

Now the registration field. Berlin requires a registration number for short-term rentals, and Airbnb asks hosts to
type it into a free-text field. We do **not** print this field: in the raw file it contains names of private
persons. Instead we count its patterns with the same function that `prepare_airbnb.py` uses.

In [ ]:
sys.path.insert(0, str(ROOT / "case-study"))
from prepare_airbnb import license_status   # the classification rule of the preparation script

raw_license = pd.read_csv(RAW / "listings.csv.gz", usecols=["license"])["license"]
status_raw = raw_license.map(license_status)
del raw_license                              # keep the names out of the notebook's memory and output
print(status_raw.value_counts().to_dict())
print("same categories as the prepared table:", status_raw.value_counts().equals(listings["license_status"].value_counts()))

check("accuracy", "raw listings", "license", "registration field holds the name of a private host",
      status_raw.eq("private host name"), n, "major")
check("accuracy", "raw listings", "license", "registration field holds the name of a company",
      status_raw.eq("legal entity name"), n, "minor")
numbers = listings.loc[listings["license_status"].eq("registration number"), "license"]
check("validity", "listings", "license", "registration number in the official format 01/Z/RA/123456-24",
      ~numbers.str.fullmatch(r"\d{2}/[A-Z]/[A-Z]{2}/\d{6}-\d{2}"), len(numbers), "minor")

1,685 hosts typed their own first and last name into the field, 2,602 the name of a company. The name of a private
person is **personal data** under the GDPR, even when it was shown on a public page. The course has no purpose that
needs it: the analysis needs only the *type* of entry. So `prepare_airbnb.py` replaces every entry that is not a
registration number by its category and adds `license_status` (**data minimisation**). That is a quality decision
as much as a legal one: the field was meant for a number, so a name is also an accuracy problem.

**TODO 2.** Do hosts with several listings (`calculated_host_listings_count > 1`) fill in the field differently
from hosts with one listing? Make a table of `license_status` shares for both groups. Which of the two statements
"multi-listing hosts are more often registered" and "multi-listing hosts more often fill in the field" does the
table support? (Session 5 tests this with a chi-square test.)

In [ ]:
# TODO 2

## 7. The report

In [ ]:
order = {"critical": 0, "major": 1, "minor": 2, "ok": 3}
report = (pd.DataFrame(results)
          .assign(rank=lambda d: d["severity"].map(order), share=lambda d: d["share"].round(4))
          .sort_values(["rank", "n_failed"], ascending=[True, False])
          .drop(columns="rank")
          .reset_index(drop=True))
pd.set_option("display.max_colwidth", 70)
report[report["severity"] != "ok"]

In [ ]:
print(f"{len(report)} checks, {(report['severity'] == 'ok').sum()} passed")
report.groupby(["dimension", "severity"]).size().unstack(fill_value=0)

In [ ]:
def to_markdown(df: pd.DataFrame) -> str:
    """A small Markdown table writer (avoids an extra dependency)."""
    lines = ["| " + " | ".join(df.columns) + " |", "|" + "---|" * len(df.columns)]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)


findings = report[report["severity"] != "ok"][["severity", "dimension", "table", "column", "check", "n_failed", "share"]]
passed = report[report["severity"] == "ok"]["check"]
text = f"""# Data quality report: Berlin Airbnb listings

Data: Inside Airbnb, Berlin, snapshot {(DATA / "snapshot.txt").read_text().strip()}: `listings` ({len(listings):,} rows),
`calendar` ({len(calendar):,} rows), `reviews_monthly` ({len(reviews_monthly):,} rows).
Checks run: {len(report)}; passed: {len(passed)}.

## Findings

{to_markdown(findings)}

## Checks that passed

{", ".join(passed)}

## Recommended actions

1. (TODO) ...
"""
print(text[:3500])

**TODO 3.** Write the *Recommended actions*: for each major and minor finding, say what should be done (remove,
correct, flag, impute, document) and why. Example: "maximum_nights = 2,147,483,647 for 2 listings: set to missing,
document; never average maximum stays without this step." Compare your actions with the cleaning log of workbook 15
in block 3.

**TODO 4.** Add two checks of your own, for example: is the district of each listing consistent with its
neighbourhood (one neighbourhood, one district)? Do listings with `availability_365 == 0` still have reviews in the
last twelve months (`number_of_reviews_ltm`)?

In [ ]:
# TODO 4

## 8. The same rules as a pandera schema

[pandera](https://pandera.readthedocs.io/) (MIT licence) describes the expected structure of a DataFrame as a
**schema**: columns, types and checks. `validate(lazy=True)` runs all checks and collects every failure instead of
stopping at the first one. Workbook 01 introduces pandera in general. pandera is part of the course environment;
otherwise start Jupyter with `uv run --with pandera jupyter lab`.

In [ ]:
try:
    import pandera.pandas as pa
except ImportError:
    pa = None
    print("pandera is not installed: start Jupyter with  uv run --with pandera jupyter lab")

if pa is not None:
    listing_schema = pa.DataFrameSchema(
        {
            "id": pa.Column(int, unique=True),
            "room_type": pa.Column(str, pa.Check.isin(sorted(ROOM_TYPES))),
            "latitude": pa.Column(float, pa.Check.in_range(*LAT)),
            "longitude": pa.Column(float, pa.Check.in_range(*LON)),
            "accommodates": pa.Column(int, pa.Check.ge(1)),
            "price": pa.Column(float, pa.Check.in_range(10, 1000), nullable=True),
            "minimum_nights": pa.Column(float, pa.Check.le(365), nullable=True),
            "maximum_nights": pa.Column(float, pa.Check.lt(2**31 - 1, error="sentinel 2,147,483,647"), nullable=True),
            "availability_365": pa.Column(int, pa.Check.in_range(0, 365)),
            "review_scores_rating": pa.Column(float, pa.Check.in_range(0, 5), nullable=True),
            "license_status": pa.Column(str, pa.Check.isin(["registration number", "missing", "legal entity name",
                                                             "private host name", "other"])),
        },
        unique=SAME_OFFER,
    )
    try:
        listing_schema.validate(listings, lazy=True)
    except pa.errors.SchemaErrors as err:
        print(err.failure_cases.groupby(["column", "check"], dropna=False).size())

## 9. Validation rules as tests

A **test** is a function that fails loudly when a rule is broken. Running the tests after every data update (locally
or in continuous integration, Session 2) turns the quality report into a guard. From the repository root:

```bash
uv run --with pytest pytest sessions/04-data-quality/workbooks/quality -v
```

Known problems of the snapshot are marked as *expected failures* (`xfail`) with the reason, so that they are
documented without making the test run fail. The test `test_registration_field_holds_no_names` encodes the privacy
rule: if a new version of the preparation script forgot to mask the field, the test would fail. After block 3
(workbook 15), the tests on the cleaned table run as well and must pass.

In [ ]:
test_file = Path("quality") / "test_listings_quality.py"
if not test_file.exists():
    test_file = ROOT / "sessions/04-data-quality/workbooks/quality/test_listings_quality.py"
print(test_file.read_text()[:2500])